In [1]:
import json
import os

In [2]:
class Book():
    def __init__(self, title, author, isbn):
        self.title = title 
        self.author = author 
        self. isbn = isbn
        self.is_available = True

    def __str__(self):## this function automatially runs when we call book
        status = "Available" if self.is_available else "Checked out"
        return f'"{self.title}" by {self.author} (ISBN : {self.isbn})'

class Members():
    def __init__(self, name, member_id):
        self.name = name 
        self. member_id = member_id
        self.borrowed_books = []

    def max_books_allowed(self):
        return 0 
        
    def __init__(self):
        limit = self.max_books_allowed()
        borrowed = len(self.max_books_allowed)
        return f"{self.name}(ID: {self.member_id} i {borrowed}/{limit} books borrowed)"

class Student(Members):
    def max_books_allowed(self):
        return 3
    
class Faculty(Members):
    def max_books_allowed(self):
        return 10

In [15]:
class Library:
    def __init__(self, data_file = "Library_data.json"):
        self.data_file = data_file
        self.books = []
        self.members = {}
        self.load_data()

#add books
    def add_book(self, title, author, isbn):
        if self.find_book(isbn) is not None:
            print("A book with this ISBN is already exists.")
            return 
        new_book = Book(title, author, isbn)
        self.books.append(new_book)
        print(f"Added: {new_book}")
        self.save_data()
##Remove books
    def remove_book(self, isbn):
        book = self.find_book(isbn)

        if book is None:
            print("Book is not found")
            return
        if not book.is_available:
            print("This book is just checked out, cant be removed")
            return
        self.books.remove(book)
        print(f"Removed : {book.title}")
        self.save_data()
##Search
    def search_books(self, keyword):
        keyword = keyword.lower()
        result = []

        for book in self.books:
            if keyword in book.title.lower() or keyword in book.author.lower():
                result.append(book)
        if len(result) ==0:
            print("No Match")
        else:
            print(f"Found {len(result)} matching book(s):")
            for book in result:
                print(f"{book}")
##list the books
    def list_books(self):
        if len(self.books) == 0:
            print("The library has no books yet")
            return
        for book in self.books:
            print(f"{book}")
##Find book
    def find_book(self, isbn):
        for book in self.books:
            if book.isbn == isbn:
                return book
        return None


    def register_member(self, name, member_id, member_type):
        if member_id in self.members:
            print("A member with this ID already exists.")
            return
 
        member_type = member_type.lower()
 
        if member_type == "student":
            new_member = Student(name, member_id)
        elif member_type == "faculty":
            new_member = Faculty(name, member_id)
        else:
            print("Member type must be 'student' or 'faculty'.")
            return
 
        self.members[member_id] = new_member
        print(f"Registered: {new_member}")
        self.save_data()
 


    
    #---------------Borrowing---------------
    def issue_book(self, isbn , member_id):
        book = self.find_book(isbn)
        member = self.members.get(member_id)

        if book is None:
            print("Book is not found with that ISBN")
            return
        if not book.is_available:
            print("Sorry book is checked out")
            return
        if len(member.borrowed_books)>=member.max_books_allowed():
            print(f"{member.name} has reached their borrowing limit")
            return

        book.is_available = False
        member.borrowed_books.append(isbn)
        print(f"{member.name} borrowed '{book.title}'.")
        self.save_data()

    def return_book(self, isbn, member_id):
        book = self.find_book(isbn)
        member = self.members.get(member_id)

        if book is None or member is None:
            print("Book or Member not found")
            return
        if isbn not in member.borrowed_books:
            print(f"{member.name} does not borrowed this book")
            return
        boo.is_available =True
        member.borrowed_books.remove(isbn)
        print(f"{member.name} return the book {bool.title}")
        self.save_data()
        
    def show_member_books(self, member_id):
        member = self.members.get(member_id)

        if member is None:
            print("invalid id")
            return
        if len(member.borrowed_books)==0:
            print(f"{member.name} doesnt borrowed any book")
            return
        print(f"Books borrowed by {member.name}:")
        for isbn in member.borrowed_books:
            book = self.find_book(isbn)
            if book is not None:
                print(f"{book.title}")

#-----------------------Save & load data 
    def save_data(self):
        books_data =  []
        for book in self.books:
            books_data.append({
                "title" : book.title,
                "author": book.author,
                "isbn" : book.isbn,
                "is_available": book.is_available
            })
        member_data = []
        for member in self.members.values():
            #isinstance() checks what type of object is
            if isinstance(member, student):
                member_type ="student"
            else: member_type ="faculty"

            members_data.append({
                "name": member.name,
                "member_id": member.member_id,
                "type": member_type,
                "borrowed_books": member.borrowed_books
            })
        data = {"books":books_data,"member": member_data}

        with open(self.data_file, "w") as f:
            json.dump(data, f, indent=2)

    def load_data(self):
        if not os.path.exists(self.data_file):
            return
        with open(self.data_file , "r")as f:
            data = json.load(f)
 # rebuild every Book object 
        for b in data.get("books", []):
            book = Book(b["title"], b["author"], b["isbn"])
            book.is_available = b["is_available"]
            self.books.append(book)
#rebuild every Member object
        for m in data.get("members", []):
            if m["type"] == "student":
                member = Student(m["name"], m["member_id"])
            else:
                member = Faculty(m["name"], m["member_id"])
            member.borrowed_books = m["borrowed_books"]
            self.members[m["member_id"]] = member
            
        

In [16]:

def print_menu():
    print("\n===== Library Menu =====")
    print("1. Add a book")
    print("2. Remove a book")
    print("3. Register a member")
    print("4. Search for a book")
    print("5. List all books")
    print("6. Issue a book")
    print("7. Return a book")
    print("8. View a member's borrowed books")
    print("9. Exit")
 
 
def main():
    library = Library()
 
    while True:
        print_menu()
        choice = input("Choose an option: ")
 
        if choice == "1":
            title = input("Book title: ")
            author = input("Author: ")
            isbn = input("ISBN: ")
            library.add_book(title, author, isbn)
 
        elif choice == "2":
            isbn = input("ISBN of the book to remove: ")
            library.remove_book(isbn)
 
        elif choice == "3":
            name = input("Member name: ")
            member_id = input("Member ID: ")
            member_type = input("Type (student/faculty): ")
            library.register_member(name, member_id, member_type)
 
        elif choice == "4":
            keyword = input("Search by title or author: ")
            library.search_books(keyword)
 
        elif choice == "5":
            library.list_books()
 
        elif choice == "6":
            isbn = input("ISBN of the book to issue: ")
            member_id = input("Member ID: ")
            library.issue_book(isbn, member_id)
 
        elif choice == "7":
            isbn = input("ISBN of the book to return: ")
            member_id = input("Member ID: ")
            library.return_book(isbn, member_id)
 
        elif choice == "8":
            member_id = input("Member ID: ")
            library.show_member_books(member_id)
 
        elif choice == "9":
            print("Goodbye! Everything's saved.")
            break
 
        else:
            print("Please enter a number between 1 and 9.")
 
 
# this makes sure main() only runs when this file is run directly,
# not if it's ever imported into another file
if __name__ == "__main__":
    main()
 


===== Library Menu =====
1. Add a book
2. Remove a book
3. Register a member
4. Search for a book
5. List all books
6. Issue a book
7. Return a book
8. View a member's borrowed books
9. Exit


Choose an option:  #


Please enter a number between 1 and 9.

===== Library Menu =====
1. Add a book
2. Remove a book
3. Register a member
4. Search for a book
5. List all books
6. Issue a book
7. Return a book
8. View a member's borrowed books
9. Exit


Choose an option:  3
Member name:  dfg
Member ID:  gfd
Type (student/faculty):  student


TypeError: Members.__init__() takes 1 positional argument but 3 were given